In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re
from itertools import chain

In [0]:
def to_null(df: DataFrame):
    for field in df.schema.fields:
        col_name = str(field.name)
        # Use F.trim() inside the check to catch " \N "
        df = df.withColumn(
            col_name,
            F.when(F.trim(F.col(col_name)) == r"\N", None)
             .when(F.col(col_name) == "null", None) # Optional: catch literal "null" strings
             .otherwise(F.col(col_name))
        )
    return df

def trim_strings(df: DataFrame):
    for field in df.schema.fields:
        col_name = str(field.name)
        if str(field.dataType) == "StringType()":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))
    return df

# ── 3. Drop url column if it exists ──────────────────────────────────────────
def drop_url(df: DataFrame):
    if "url" in df.columns:
        df = df.drop("url")
    return df

# ── 4. Cast *_date columns that match yyyy-MM-dd to DateType ─────────────────
def cast_date_columns(df: DataFrame):
    for field in df.schema.fields:
        # Check for string columns with 'date' or 'dob' in the name
        if isinstance(field.dataType, StringType) and ("date" in field.name or "dob" in field.name):
            # Step 1: Parse the string into a DateType using multiple possible source formats
            # Step 2: Use date_format to force the output to 'yyyy-MM-dd'
            df = df.withColumn(
                field.name, 
                F.date_format(
                    F.coalesce(
                        F.to_date(F.col(field.name), "yyyy-MM-dd"),
                        F.to_date(F.col(field.name), "M/d/yy"),
                        F.to_date(F.col(field.name), "MM/dd/yyyy"),
                        F.to_date(F.col(field.name), "yyyy/MM/dd")
                    ), 
                    "yyyy-MM-dd"
                )
            )
    return df

# ── 5. Cast to IntegerType (after \N nulling) ───────────────────
def cast_to_integer(df: DataFrame):
    columns = ["milliseconds", "position", "number", "grid", "fastest lap", "rank"]
    for col_name in columns:
        if col_name in [str(c) for c in df.columns]:
            df = df.withColumn(col_name, F.expr(f"try_cast(`{col_name}` AS INT)"))
    return df

def position_text_fix(df: DataFrame):
    if "positionText" in df.columns:
        status_map = {
            "R": "Retired",
            "D": "Disqualified",
            "E": "Excluded",
            "F": "Failed to qualify",
            "N": "Not Classified",
            "W": "Withdrawn",
            "-": "None"
        }
        map_expr = F.create_map([F.lit(x) for x in chain(*status_map.items())])
        df = df.withColumn(
            "positionText",
            F.coalesce(map_expr[F.col("positionText")], F.col("positionText"))
        )
    return df

def transform_time_to_precision(df: DataFrame) -> DataFrame:
    # Force column names to strings to avoid Column object issues
    cols = [str(c) for c in df.columns]
    cols_lower = [c.lower() for c in cols]

    has_time_col = any("time" in c and c != "fastestlaptime" for c in cols_lower)

    if all([("duration" in cols_lower), has_time_col, ("milliseconds" in cols_lower)]):
        duration_cols = [c for c in cols if "duration" in c.lower()]
        df = df.drop(*duration_cols)
        cols = [str(c) for c in df.columns]
        cols_lower = [c.lower() for c in cols]

    elif "milliseconds" in cols_lower:
        time_cols_to_drop = [
            c for c in cols
            if "time" in c.lower()
            and c.lower() != "fastestlaptime"
            and c.lower() != "milliseconds"
        ]
        if time_cols_to_drop:
            df = df.drop(*time_cols_to_drop)
            cols = [str(c) for c in df.columns]
            cols_lower = [c.lower() for c in cols]

    targets = ["time", "duration", "q1", "q2", "q3", "fastest"]

    for col_name in cols:
        low_col = col_name.lower()

        if any(word in low_col for word in targets):
            if low_col in ["milliseconds", "fastestlap"]:
                continue

            df = df.withColumn(col_name, F.regexp_replace(F.col(col_name), r"^\+", ""))

            df = df.withColumn(
                col_name,
                F.when(~F.col(col_name).contains(":"), F.concat(F.lit("0:"), F.col(col_name)))
                 .otherwise(F.col(col_name))
            )
            df = df.withColumn(col_name, F.when(F.col(col_name).rlike(r'\\N'), None).otherwise(F.col(col_name)))
            parts = F.split(F.col(col_name), "[:.]")
            df = df.withColumn(
                col_name,
                (F.coalesce(parts.getItem(0).cast("int"), F.lit(0)) * 60000) +
                (F.coalesce(parts.getItem(1).cast("int"), F.lit(0)) * 1000) +
                (F.coalesce(parts.getItem(2).cast("int"), F.lit(0)))
            ).withColumn(col_name, F.col(col_name).cast("int"))

    return df

In [0]:
# Master function
def apply_general_silver_transforms(df: DataFrame):
    df = df.replace('\\N', None)
    df = to_null(df)
    df = trim_strings(df)
    df = drop_url(df)
    df = cast_date_columns(df)
    df = cast_to_integer(df)
    df = position_text_fix(df)
    df = transform_time_to_precision(df)
    return df